# Phase 3: Predictive Modeling

One model per observation age (not one global model), predicting `future_paid`
-- dollars still to come -- from the claim's state at the snapshot. Ages
**4** (1 year out) and **8** (2 years out), one on each side of the time split.

**This notebook works through models from simplest to most complex, in
order:** chain ladder (the actuarial standard, no ML at all) sets the bar
-> linear regression is the simplest possible ML model, just to see if ML
helps at all -> LightGBM, which wins by a wide margin -> hyperparameter
tuning and Tweedie, two honest attempts to push LightGBM further -> then a
ceiling test that asks whether 0.237 is actually a bad score or close to the
best this data can support. Limitations pulls it together at the end.

**Final model: LightGBM, the 9 features below, raw-dollar target, predictions
clipped at zero. Age-4 test R² = 0.237.** Neither tuning nor Tweedie beat it
-- see "Limitations" for why, and why that's a real finding.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
from snapshot_builder import build_snapshot_triangle, ceil_period, load_truth, CUTOFF
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
from lightgbm import LGBMRegressor


df = pd.read_csv('../data/synthetic_transactions_with_covariates.csv')
snapshot_triangle = build_snapshot_triangle(df, cutoff=CUTOFF, step=1)
snapshot_triangle.head()

portfolio: 3624 claims
excluded (notified at or after the last grid point, 40): 301 (8.3%) -- no room left to show any development


snapshot triangle: 382485 rows (379840 observed, 2645 settled-exit), 3323 claims represented


,claim_no,snapshot_period,observation_age,observation_period,row_status,future_paid,paid_to_date,is_settled_at_obs
0,1,2,1.0,3.0,observed,0.0,0.0,False
1,1,2,2.0,4.0,observed,0.0,0.0,False
2,1,2,3.0,5.0,observed,0.0,0.0,False
3,1,2,4.0,6.0,observed,0.0,0.0,False
4,1,2,5.0,7.0,observed,0.0,0.0,False


## Add features

The minimal triangle only has `claim_no`, `snapshot_period`, `paid_to_date`,
etc. -- no covariates, no payment-history detail. This project demos the
*mechanism* (snapshot triangle -> ML -> clustering -> chain ladder), not a
feature-engineering exercise, so the feature list below is kept small and
each one earns its place for a specific reason rather than throwing in
everything available.

Static features (fixed at notification, don't change snapshot to snapshot)
give the model what a claims handler would know on day one. Dynamic features
(recomputed at every snapshot) give it a read on how the claim is behaving
*so far* -- paid_to_date, how long it's been open, whether payments are
speeding up or slowing down. A model with only static features can't tell an
old, winding-down claim apart from a young one that looks identical on paper;
the dynamic features are what provides that insight.

Final covariates:
| feature | axis | why |
|---|---|---|
| `injury_severity` | static | how inherently large this claim is (one-hot encode -- sev 6 has a *lower* effect than sev 1, see CLAUDE.md) |
| `age_of_claimant` | static | one of SynthETIC's 3 built-in risk covariates (with severity/legal rep) -- tested against the other 8 features in Phase 4 and found a small, mixed effect (-0.011 age4 test R², +0.008 age8), included for consistency with the covariate set Phase 4 clusters on, not because it drives accuracy |
| `legal_representation` | static | litigated claims develop longer, pay more |
| `notidel` | static | late-reported claims behave differently -- classic reserving predictor |
| `paid_to_date` | dynamic | scale -- already a native column on the core triangle, no extra computation needed |
| `periods_since_notification` | dynamic | how far into development we're standing |
| `periods_since_last_payment` | dynamic | still active vs winding down |
| `has_any_payment` | dynamic flag | the other dynamic features are undefined before the first payment -- this lets the model tell "no payments yet" apart from "small payments" |
| `payment_acceleration` | dynamic | "unpredictable dynamic covariate" -- used as-of-snapshot, never forecast forward. This is the column that most directly demonstrates the paper's central claim |

Banned during Machine Learning testing to prevent bad learning: `claim_size`, `setldel`,
`settlement_period`, `outstanding_at_snapshot`, `ultimate_observed`,
`payment_inflated`, `is_settled_at_obs`.

In [2]:
from feature_engineering import add_features, FEATURE_COLUMNS

snapshot_triangle, claims = add_features(df, snapshot_triangle)
snapshot_triangle.head()

,claim_no,snapshot_period,observation_age,observation_period,row_status,future_paid,paid_to_date,is_settled_at_obs,injury_severity,legal_representation,notidel,age_of_claimant,periods_since_notification,periods_since_last_payment,payment_acceleration,has_any_payment
0,1,2,1.0,3.0,observed,0.0,0.0,False,1,Y,0.93,50-65,0,NaN,NaN,False
1,1,2,2.0,4.0,observed,0.0,0.0,False,1,Y,0.93,50-65,0,NaN,NaN,False
2,1,2,3.0,5.0,observed,0.0,0.0,False,1,Y,0.93,50-65,0,NaN,NaN,False
3,1,2,4.0,6.0,observed,0.0,0.0,False,1,Y,0.93,50-65,0,NaN,NaN,False
4,1,2,5.0,7.0,observed,0.0,0.0,False,1,Y,0.93,50-65,0,NaN,NaN,False


## Setup

Only `row_status == "observed"` rows have a real `future_paid` target (settled
rows are `None` -- exit markers, not training data).

**The split is by time** (`snapshot_period <= 28` train / `> 28` test):
standing at a snapshot and predicting forward is the actual job, so this is the
only score reported anywhere below.

Everything is scored through one `evaluate()` function so every number in the
notebook is directly comparable. It always scores in **dollars** and always
clips predictions at zero -- both LightGBM and LinearRegression emit negative
predictions, and a claim cannot pay a negative amount.

`bias_pct` is total predicted vs total actual dollars. R² and MAE say nothing
about whether the reserve *adds up*, which is the thing Phase 5 depends on.

In [3]:
# One row per (claim, snapshot) at observation age 4, real observed rows only.
age4 = snapshot_triangle[
    (snapshot_triangle['observation_age'] == 4) &
    (snapshot_triangle['row_status'] == 'observed')]

age8 = snapshot_triangle[
    (snapshot_triangle['observation_age'] == 8) &
    (snapshot_triangle['row_status'] == 'observed')]


def is_train(frame):
    """Train on snapshots up to period 28, test on everything after.

    Split by TIME, not randomly. A random split would put the same claim in
    both halves, letting its own future payments leak into its training
    features -- and it would also let the model learn from calendar periods
    that hadn't happened yet."""
    return frame['snapshot_period'] <= CUTOFF - 12


# create a mask to filter out rows later
train_mask_4 = is_train(age4)
train_mask_8 = is_train(age8)

# The right answer - what actually got paid out
actual_4 = age4['future_paid'].astype(float)
actual_8 = age8['future_paid'].astype(float)

print(f'{len(age4):,} age-4 rows -- {train_mask_4.sum():,} train (snapshot <= 28), '
    f'{(~train_mask_4).sum():,} test (snapshot > 28)')
print(f'{len(age8):,} age-8 rows -- {train_mask_8.sum():,} train (snapshot <= 28), '
    f'{(~train_mask_8).sum():,} test (snapshot > 28)')

20,784 age-4 rows -- 14,951 train (snapshot <= 28), 5,833 test (snapshot > 28)
17,878 age-8 rows -- 14,951 train (snapshot <= 28), 2,927 test (snapshot > 28)


In [4]:
def score(name, actual_train, pred_train, actual_test, pred_test):
    """Every model in this notebook is measured the same way, so the numbers
    are directly comparable.

    bias_pct = total predicted dollars vs total actual dollars. R2 and MAE
    both ignore whether the money adds up, which is what reserving cares about.
    """
    return dict(model=name,
                train_r2=r2_score(actual_train, pred_train),
                test_r2=r2_score(actual_test, pred_test),
                test_mae=mean_absolute_error(actual_test, pred_test),
                bias_pct=(pred_test.sum() / actual_test.sum() - 1) * 100)


def show(rows):
    """Print a list of score() results as one table."""
    print(pd.DataFrame(rows).to_string(index=False, formatters={
        'train_r2': '{:.3f}'.format, 'test_r2': '{:.3f}'.format,
        'test_mae': '${:,.0f}'.format, 'bias_pct': '{:+.1f}%'.format}))

## Simplified Chain-Ladder for individual claim record baseline

Before trying any ML, set the bar with the method actuaries already use:
chain ladder. If a plain regression or LightGBM can't beat this, ML isn't
adding anything. This version is a simplified, per-claim-row take on it
(real chain ladder runs on aggregated triangles, not individual claims) --
close enough to be a fair baseline, not a production implementation.

In [5]:
def chain_ladder(frame, by=None):
    """Predict future payments as:  paid_to_date x (growth factor - 1)

    Main idea: look at how much claims grew in the past, and assume new claims
    grow the same way.

    by=None  -> one growth factor for every claim
    by='col' -> a separate growth factor for each value of that column
    """
    train = frame[is_train(frame)]        # create training data

    before = train['paid_to_date'].sum()          # money paid as of the snapshot
    after = before + train['future_paid'].sum()   # ...and one observation window later
    overall_growth = after / before               # e.g. 1.5 -> claims grow 50%

    if by is None:
        growth = pd.Series(overall_growth, index=frame.index)
    else:
        # same formula, but computed separately inside each group
        sums = train.groupby(by)[['paid_to_date', 'future_paid']].sum()
        table = (sums['paid_to_date'] + sums['future_paid']) / sums['paid_to_date']
        # look up each row's own factor; fall back to overall if the group is unseen
        growth = frame[by].map(table).fillna(overall_growth)

    # "- 1" because we predict the NEW money, not the new total:
    # $10,000 paid x 1.5 growth = $15,000 total, so $5,000 is still to come
    return (frame['paid_to_date'] * (growth - 1)).clip(lower=0)


results = []
for name, by in [('chain ladder (one factor)', None),
                ('chain ladder (by maturity)', 'periods_since_notification')]:  # Grouping by periods_since_notification represents how old the claim itself is at that snapshot (how many quarters since it was first reported)
    pred = chain_ladder(age4, by)         # computed once, then sliced two ways
    results.append(score(name,
                        actual_4[train_mask_4],  pred[train_mask_4],
                        actual_4[~train_mask_4], pred[~train_mask_4]))
show(results)

                     model train_r2 test_r2 test_mae bias_pct
 chain ladder (one factor)   -0.973  -0.590 $106,213    +8.8%
chain ladder (by maturity)   -0.429  -0.194  $96,525    -5.8%


## Linear Regression Model

Fits one straight line: `future_paid = a1*feature1 + a2*feature2 + ... + b`.
The simplest possible model -- included as a baseline to see whether the
smarter models (LightGBM) are actually worth the extra complexity.

Categorical columns (`injury_severity`, `legal_representation`, `age_of_claimant`) get one-hot
encoded -- turned into 0/1 columns, since LinearRegression can't read text
directly. The two columns that can be blank (`periods_since_last_payment`,
`payment_acceleration` -- both undefined before a claim's first payment) get
filled with 0; `has_any_payment` tells the model when to trust that 0 and when
it means something else.

In [6]:
categorical_cols = ['injury_severity', 'legal_representation', 'age_of_claimant']
nan_prone_cols = ['periods_since_last_payment', 'payment_acceleration']


def make_preprocessor(impute=True):
    # a FRESH transformer every time -- Pipeline fits its steps in place, so
    # reusing one instance across two different models lets the second .fit()
    # silently overwrite the first model'''s fitted state
    #
    # impute=True  -> fill blanks with 0 (LinearRegression crashes on NaN)
    # impute=False -> leave blanks as NaN (LightGBM handles missing values natively 
    fill_step = [('impute', SimpleImputer(strategy='constant', fill_value=0), nan_prone_cols)] if impute else []
    return ColumnTransformer(transformers=[
        ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
        *fill_step,
    ], remainder='passthrough')

linreg = Pipeline([('prep', make_preprocessor()), ('model', LinearRegression())])
linreg.fit(age4.loc[train_mask_4, FEATURE_COLUMNS], actual_4[train_mask_4]) # train on data and learn to predict

pred_lin = np.clip(linreg.predict(age4[FEATURE_COLUMNS]), 0, None)  # make predictions on all data amd force predictions to be $0

results.append(score('''linear regression''',
                    actual_4[train_mask_4],  pred_lin[train_mask_4],
                    actual_4[~train_mask_4], pred_lin[~train_mask_4]))
show(results)

                     model train_r2 test_r2 test_mae bias_pct
 chain ladder (one factor)   -0.973  -0.590 $106,213    +8.8%
chain ladder (by maturity)   -0.429  -0.194  $96,525    -5.8%
         linear regression    0.073   0.043  $88,856   -10.2%


## LightGBM Model

Instead of one straight line for the whole dataset, LightGBM builds decision
trees -- splitting on rules like "if `paid_to_date` > $5,000, go one way,
otherwise go another." That lets it capture bends and interactions a straight
line can't, and it isn't thrown off by a few huge outlier claims the way
LinearRegression is.

In [7]:
def lgbm(**kw):
    return LGBMRegressor(random_state=20200131, verbose=-1, **kw)


lgb_model = Pipeline([('prep', make_preprocessor(impute=False)),  # let LightGBM see the real NaNs
                    ('model', lgbm())])
lgb_model.fit(age4.loc[train_mask_4, FEATURE_COLUMNS], actual_4[train_mask_4])

pred_lgb = np.clip(lgb_model.predict(age4[FEATURE_COLUMNS]), 0, None)

results.append(score('LightGBM',
                    actual_4[train_mask_4],  pred_lgb[train_mask_4],
                    actual_4[~train_mask_4], pred_lgb[~train_mask_4]))
show(results)

                     model train_r2 test_r2 test_mae bias_pct
 chain ladder (one factor)   -0.973  -0.590 $106,213    +8.8%
chain ladder (by maturity)   -0.429  -0.194  $96,525    -5.8%
         linear regression    0.073   0.043  $88,856   -10.2%
                  LightGBM    0.766   0.237  $73,994    -4.7%


## Hyperparameter Tuning (Random Search)

LightGBM clearly won above, so the next honest question is: can *tuning* it
push performance further? Random search widens 4 knobs -- `num_leaves`,
`learning_rate`, `n_estimators`, `min_child_samples` -- into ranges and
randomly samples combinations to try, rather than hand-picking a few.

Tuning gets its own split so the real test set never gets touched:
- train on snapshot <= 20
- score each random combination on snapshot 21-28
- whichever wins gets retrained on all of snapshot <= 28 and checked once,
  for real, on snapshot > 28 (next cell)

In [8]:
import random
random.seed(20200131)  # reproducible 

tune_train = age4['snapshot_period'] <= 20
tune_val = (age4['snapshot_period'] > 20) & (age4['snapshot_period'] <= 28)

N_TRIALS = 20  # how many random combinations to try -- more = better coverage, but slower

tuning_rows = []
for _ in range(N_TRIALS):
    params = dict(
        num_leaves=random.randint(10, 50),
        learning_rate=round(random.uniform(0.01, 0.3), 3),
        n_estimators=random.randint(50, 500),
        min_child_samples=random.randint(5, 100),
    )
    model = Pipeline([('prep', make_preprocessor(impute=False)), ('model', lgbm(**params))])
    model.fit(age4.loc[tune_train, FEATURE_COLUMNS], actual_4[tune_train])
    pred_val = np.clip(model.predict(age4.loc[tune_val, FEATURE_COLUMNS]), 0, None)
    val_r2 = r2_score(actual_4[tune_val], pred_val)
    tuning_rows.append({**params, 'val_r2': round(val_r2, 3)})

tuning_df = pd.DataFrame(tuning_rows).sort_values('val_r2', ascending=False)
print(f'{len(tuning_df)} random combinations tried\n')
print(tuning_df.head(10).to_string(index=False))

20 random combinations tried

 num_leaves  learning_rate  n_estimators  min_child_samples  val_r2
         20          0.224           195                 50   0.245
         45          0.091           204                 49   0.244
         38          0.142           177                 62   0.240
         43          0.175           236                 65   0.235
         50          0.017           251                 37   0.231
         29          0.184           335                 57   0.229
         46          0.106           259                 61   0.227
         21          0.031           421                 87   0.224
         27          0.031           259                 87   0.223
         14          0.140           492                 69   0.218


In [9]:
# whichever combo won on the validation slice, retrain on all of the real
# training data (snapshot <= 28) and check ONCE against the real test set
PARAM_COLS = ['num_leaves', 'learning_rate', 'n_estimators', 'min_child_samples']
best_params = tuning_df.iloc[0][PARAM_COLS].to_dict()
for int_col in ['num_leaves', 'n_estimators', 'min_child_samples']:
    best_params[int_col] = int(best_params[int_col])
print(f'best combo: {best_params}')

tuned_model = Pipeline([('prep', make_preprocessor(impute=False)), ('model', lgbm(**best_params))])
tuned_model.fit(age4.loc[train_mask_4, FEATURE_COLUMNS], actual_4[train_mask_4])
pred_tuned = np.clip(tuned_model.predict(age4[FEATURE_COLUMNS]), 0, None)

results.append(score('LightGBM (tuned)',
                    actual_4[train_mask_4],  pred_tuned[train_mask_4],
                    actual_4[~train_mask_4], pred_tuned[~train_mask_4]))
show(results)

best combo: {'num_leaves': 20, 'learning_rate': 0.224, 'n_estimators': 195, 'min_child_samples': 50}
                     model train_r2 test_r2 test_mae bias_pct
 chain ladder (one factor)   -0.973  -0.590 $106,213    +8.8%
chain ladder (by maturity)   -0.429  -0.194  $96,525    -5.8%
         linear regression    0.073   0.043  $88,856   -10.2%
                  LightGBM    0.766   0.237  $73,994    -4.7%
          LightGBM (tuned)    0.813   0.203  $74,589    -6.2%


## Tweedie

Payments run from $0 up into the millions, and a few giant claims carry a huge
share of the total dollars. Tweedie is a LightGBM setting built for exactly
this shape -- a target that can be zero plus a long positive tail. It's the
standard actuarial choice for this kind of data, so worth testing even though
it's only ~9% zeros here, not the 50%+ Tweedie usually expects.

Same model, same features, same everything as the LightGBM above -- only the
`objective` changes.

In [10]:
pct_zero = (age4['future_paid'] == 0).mean() * 100
print(f'{pct_zero:.1f}% of rows have future_paid == exactly $0')

# our number is low because a row only exists in age4 for a claim that's currently open at that snapshot — between when it was reported and when it settled. Once a claim settles, it drops out.
# In other words, we're only ever looking at claims that are actively being tracked, mid-life. 
# And an open claim getting exactly $0 over an entire year (4 quarters) is fairly rare — claims usually have something moving, even if small, while they're still open.

8.7% of rows have future_paid == exactly $0


In [11]:
tweedie_model = Pipeline([('prep', make_preprocessor(impute=False)),  # still LightGBM under the hood
                        ('model', lgbm(objective='tweedie', tweedie_variance_power=1.3))])
tweedie_model.fit(age4.loc[train_mask_4, FEATURE_COLUMNS], actual_4[train_mask_4])

pred_tw = np.clip(tweedie_model.predict(age4[FEATURE_COLUMNS]), 0, None)

results.append(score('tweedie',
                    actual_4[train_mask_4],  pred_tw[train_mask_4],
                    actual_4[~train_mask_4], pred_tw[~train_mask_4]))
show(results)

                     model train_r2 test_r2 test_mae bias_pct
 chain ladder (one factor)   -0.973  -0.590 $106,213    +8.8%
chain ladder (by maturity)   -0.429  -0.194  $96,525    -5.8%
         linear regression    0.073   0.043  $88,856   -10.2%
                  LightGBM    0.766   0.237  $73,994    -4.7%
          LightGBM (tuned)    0.813   0.203  $74,589    -6.2%
                   tweedie    0.817   0.156  $67,925   -25.7%


## Is 0.237 actually low? Testing the ceiling

Training R² looks fine for LightGBM and Tweedie, but test R² is a different
story -- both are pretty lackluster once you predict on unseen data. Tuning
and Tweedie were two honest attempts to close that gap and neither worked
(see the sections above), which raises the real question: is 0.237 a weak
*model*, or close to the best any model could do with this information? To
find out, build one "cheating" version that gets to see two things it
normally can't --

- **outstanding_at_snapshot** -- the claim's true final cost minus what it has
  paid so far (i.e. exactly how much money is truly left)
- **periods_to_settlement** -- how many quarters until the claim actually closes

Both come straight from the answer key (`claim_size`, `setldel`), so this model
is **diagnostic only** -- it exists purely to measure the ceiling, and neither
column is ever allowed into a real model.

In [12]:
# --- build the two cheat columns, on a COPY so age4 itself stays clean -------
claim_lookup = claims.set_index('claim_no')
true_settlement = (claim_lookup['occurrence_time'] + claim_lookup['notidel'] + claim_lookup['setldel']).apply(ceil_period)

age4_oracle = age4.copy()
age4_oracle['outstanding_at_snapshot'] = (
    age4_oracle['claim_no'].map(claim_lookup['claim_size']) - age4_oracle['paid_to_date'])
age4_oracle['periods_to_settlement'] = (
    age4_oracle['claim_no'].map(true_settlement) - age4_oracle['snapshot_period'])

ORACLE_COLUMNS = FEATURE_COLUMNS + ['outstanding_at_snapshot', 'periods_to_settlement']

oracle_model = Pipeline([('prep', make_preprocessor(impute=False)),
                        ('model', lgbm())])
oracle_model.fit(age4_oracle.loc[train_mask_4, ORACLE_COLUMNS], actual_4[train_mask_4])
pred_oracle = np.clip(oracle_model.predict(age4_oracle[ORACLE_COLUMNS]), 0, None)

results.append(score('oracle (cheats -- ceiling only, never a real model)',
                    actual_4[train_mask_4],  pred_oracle[train_mask_4],
                    actual_4[~train_mask_4], pred_oracle[~train_mask_4]))
show(results)

                                              model train_r2 test_r2 test_mae bias_pct
                          chain ladder (one factor)   -0.973  -0.590 $106,213    +8.8%
                         chain ladder (by maturity)   -0.429  -0.194  $96,525    -5.8%
                                  linear regression    0.073   0.043  $88,856   -10.2%
                                           LightGBM    0.766   0.237  $73,994    -4.7%
                                   LightGBM (tuned)    0.813   0.203  $74,589    -6.2%
                                            tweedie    0.817   0.156  $67,925   -25.7%
oracle (cheats -- ceiling only, never a real model)    0.971   0.778  $16,188    -3.6%


## A second, genuinely honest test: predicting past period 40

Every test score so far -- even the "honest" 0.237 -- has one thing in common:
it's still scored using data from *inside* the visible window (payment_period
<= 40). That's a legitimate time-based test (the model never trains on the
snapshots it's tested on), but it isn't the same as checking predictions
against payments that were still in the future when the model made its guess.

The dataset actually supports that stronger check. SynthETIC simulates each
claim's *entire* life, not just the part before the pretend "today" -- the raw
file has payments past period 40 sitting right there, deliberately excluded
from everything above via `load_visible()`. `load_truth()` returns that full,
uncensored data instead. Using it here, for the first time in this notebook,
answers a sharper question: for a claim standing at snapshot 37, 38, or 39
(so its next-year window lands at period 41, 42, or 43 -- genuinely past the
cutoff), does the already-trained model's prediction hold up against what
*actually* got paid?

In [13]:
truth = load_truth(df)  # full uncensored data -- same file, but now used for real

payments_truth_by_claim = {
    claim_no: g[['payment_period', 'payment_size']].values
    for claim_no, g in truth.groupby('claim_no')
}

def cum_paid_truth(claim_no, p):
    payments = payments_truth_by_claim.get(claim_no, np.empty((0, 2)))
    return payments[payments[:, 0] <= p][:, 1].sum()

FUTURE_SNAPSHOTS = [37, 38, 39]  # snapshot + 4 = 41, 42, 43 -- genuinely past the cutoff

# one row per (claim, snapshot), features only -- reusing whatever age happens
# to already exist in the triangle at these snapshots, since features don't
# depend on which age column they were originally attached to
future_rows = snapshot_triangle[
    (snapshot_triangle['snapshot_period'].isin(FUTURE_SNAPSHOTS)) &
    (snapshot_triangle['row_status'] == 'observed')
].drop_duplicates(subset=['claim_no', 'snapshot_period'])[['claim_no', 'snapshot_period'] + FEATURE_COLUMNS].copy()

future_rows['true_future_paid'] = [
    cum_paid_truth(c, s + 4) - cum_paid_truth(c, s)
    for c, s in zip(future_rows['claim_no'], future_rows['snapshot_period'])
]

pred_future = np.clip(lgb_model.predict(future_rows[FEATURE_COLUMNS]), 0, None)

print(f"{len(future_rows):,} claims tested against real payments the model never saw, "
    f"at periods {[s + 4 for s in FUTURE_SNAPSHOTS]}")

results.append(score('LightGBM (genuine future, periods 41-43)',
                    actual_4[train_mask_4], pred_lgb[train_mask_4],
                    future_rows['true_future_paid'], pred_future))
show(results)

2,277 claims tested against real payments the model never saw, at periods [41, 42, 43]
                                              model train_r2 test_r2 test_mae bias_pct
                          chain ladder (one factor)   -0.973  -0.590 $106,213    +8.8%
                         chain ladder (by maturity)   -0.429  -0.194  $96,525    -5.8%
                                  linear regression    0.073   0.043  $88,856   -10.2%
                                           LightGBM    0.766   0.237  $73,994    -4.7%
                                   LightGBM (tuned)    0.813   0.203  $74,589    -6.2%
                                            tweedie    0.817   0.156  $67,925   -25.7%
oracle (cheats -- ceiling only, never a real model)    0.971   0.778  $16,188    -3.6%
           LightGBM (genuine future, periods 41-43)    0.766   0.003  $77,736   +21.7%


**What this second test actually shows.** The genuine-future score is far worse
than the standard test: R² near zero (no better than guessing the portfolio
average), and bias that flips sign and roughly quadruples (+21.7%, badly
over-predicting, versus -4.7% normally).

Part of this is a real, honest limitation surfacing -- but part of it is also
a fair caveat about *what's being measured*. Claims still open at snapshot 37,
38, or 39 aren't a random slice of the portfolio -- they're specifically the
ones that have survived nearly to the edge of the visible window without
settling: a systematically different (and much smaller -- 2,277 claims here
vs. 5,833 in the standard test) population than the usual test set, which
spans claims at every snapshot from 29 through 40. Training data (snapshot
<= 28) also has relatively few examples of claims still open this late, so
the model has seen less of exactly this slice.

So the honest reading isn't "0.237 was a lie and 0.003 is the truth" -- it's
that whatever the model learned generalizes worse to this specific, thin,
late-stage population than to the broader mix the standard test set covers.
That's a real, useful thing to know, and it's exactly the kind of gap a
same-window time-split test can hide.

## Limitations

**The fit is weak, and the ceiling test says most of that gap is real.** LightGBM on the 9 honest features reaches test R² = 0.237. Feeding the
same model true settlement date and true outstanding (features no one has on
the actual day) pushes that to 0.778. The missing piece is knowing *when* a
claim will settle -- something the 9 honest features only hint at.

**What was tried to close that gap, and didn't:**

- **Hyperparameter tuning** -- random search over 20 combinations of
  `num_leaves`, `learning_rate`, `n_estimators`, `min_child_samples`, picked
  on a validation slice carved out of the training period so the real test
  set stayed untouched. Test R² went 0.237 -> 0.203 -- *worse*, not better.
  The likely reason: the validation slice (snapshot 21-28) is small enough
  that whichever combination happens to fit its particular noise best isn't
  necessarily the combination that generalizes -- a real risk with a wide
  random search and a modest validation set, not a bug in the method.
- **Tweedie** (built for a target with a big spike at $0) -- doesn't fit here,
  since only 8.7% of rows are exactly $0. Test R² = 0.156, worse than plain
  LightGBM, and bias got much worse too (-25.7%).
  
  In summary, the issue is that our model does not have enough sufficient information
  (true settlement timing), and not model configuration. Closing it for real would need
a model built specifically to handle "this claim hasn't settled yet"
(survival analysis is the standard tool for that), which is a bigger build
than this scoping project demo covers.
